[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shriramnag/NeonVido-AI/blob/main/colab/NeonVido_Server.ipynb)

# 🎬 NeonVido-AI (v1.0.002) — Low-RAM Crash-Free Server
**निर्देश:** पहले ऊपर मेनू में `Runtime > Change runtime type > T4 GPU` चुनें, फिर नीचे दिए गए Play बटन को दबाएं।

In [ ]:
# 1. ज़रूरी लाइब्रेरी और 8-bit मेमोरी ऑप्टिमाइज़र (bitsandbytes) इंस्टॉल करें
!pip install -q -U diffusers transformers accelerate bitsandbytes torch fastapi uvicorn nest-asyncio imageio[ffmpeg] huggingface_hub opencv-python-headless
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import os, re, gc, time, threading, subprocess, torch, cv2
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
from diffusers import WanPipeline
from transformers import UMT5EncoderModel, BitsAndBytesConfig
from diffusers.utils import export_to_video
import uvicorn

# पुरानी RAM साफ़ करें
gc.collect()
torch.cuda.empty_cache()

# 2. 8-bit में Text Encoder लोड करें ताकि 12.7GB System RAM क्रैश न हो
print("⏳ 1/2: मेमोरी-सेफ टेक्स्ट एन्कोडर लोड हो रहा है...")
model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
quant_config = BitsAndBytesConfig(load_in_8bit=True)

text_encoder = UMT5EncoderModel.from_pretrained(
    model_id,
    subfolder="text_encoder",
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print("⏳ 2/2: Wan 2.1 वीडियो इंजन लोड हो रहा है...")
t2v_pipe = WanPipeline.from_pretrained(
    model_id,
    text_encoder=text_encoder,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)
t2v_pipe.enable_model_cpu_offload()
t2v_pipe.enable_vae_tiling()
print("✅ वीडियो इंजन बिना क्रैश हुए तैयार है!")

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class VideoReq(BaseModel):
    model: str
    prompt: str
    target_minutes: float
    ratio: str
    extend: bool = False
    use_voice: bool = False
    voice_script: str = ""

@app.post("/generate")
async def generate(req: VideoReq):
    w, h = (480, 832) if req.ratio == "9:16" else (832, 480)
    base_video = "/content/current_master.mp4"

    loops = 1 if req.target_minutes <= 0.2 else min(int((req.target_minutes * 60) // 5), 120)
    if req.extend and os.path.exists(base_video):
        loops = 1

    clip_files = []
    if req.extend and os.path.exists(base_video):
        clip_files.append(base_video)

    prompts_list = [p.strip() for p in req.prompt.split("\n") if p.strip()]

    for i in range(loops):
        current_prompt = prompts_list[i % len(prompts_list)]
        print(f"🎬 क्लिप {i+1}/{loops} बन रही है: {current_prompt}")
        frames = t2v_pipe(
            prompt=current_prompt,
            negative_prompt="blurry, low quality, distorted, watermark",
            height=h, width=w, num_frames=49, num_inference_steps=20
        ).frames[0]
        part_path = f"/content/part_{i}.mp4"
        export_to_video(frames, part_path, fps=16)
        clip_files.append(part_path)
        gc.collect()
        torch.cuda.empty_cache()

    with open("/content/list.txt", "w") as f:
        for c in clip_files:
            f.write(f"file '{c}'\n")

    subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "/content/list.txt", "-c", "copy", base_video], check=True)
    print("✅ फाइनल एक्सटेंडेड वीडियो तैयार है!")
    return FileResponse(base_video, media_type="video/mp4", filename="NeonVido-AI.mp4")

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000)

threading.Thread(target=run_api, daemon=True).start()
time.sleep(2)

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n🚀 आपका NEONVIDO-AI COLAB URL:\n" + m.group(0) + "\n")
        break